In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np

In [ ]:
x1 = np.array([1, 2, 3])
x2 = 2*x1

y = np.array([4, 6, 8])

In [ ]:
all_ones = np.ones(x1.shape[0])
X = np.array([all_ones, x1, x2]).T

In [ ]:
X.shape

In [ ]:
X

In [ ]:
def solve_normal_equation(X, y):
    try:
        theta = np.linalg.inv(X.T @ X) @ X.T @ y
        return theta
    except np.linalg.LinAlgError:
        print('The matrix is singular')
        print("X.T @ X = \n", X.T @ X)
        return None
    
### Assignment question: Use np.linalg.solve instead of inv. Why is this better?

In [ ]:
solve_normal_equation(X, y)

In [ ]:
np.linalg.matrix_rank(X), np.linalg.matrix_rank(X.T @ X)

In [ ]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()

data = np.array([x1, x2]).T

lr.fit(data, y)
lr.coef_, lr.intercept_


# Assignment question: figure why sklearn is able to solve the problem

In [ ]:
# Regularization

eps = 1e-5
X = np.array([all_ones, x1, x2]).T
X = np.eye(3)*eps + X
X

In [ ]:
np.linalg.matrix_rank(X)

In [ ]:
solve_normal_equation(X, y)

In [ ]:
# Drop variables
X = np.array([all_ones, x1]).T
print(X)

In [ ]:
solve_normal_equation(X, y)

In [ ]:
# Dummy variables

## dataset
num_records = 12
windspeed = np.random.randint(0, 10, num_records)
vehicles = np.random.randint(100, 500, num_records)
direction = np.random.choice(['N', 'S', 'E', 'W'], num_records)
pollution = np.random.randint(0, 100, num_records)

df = pd.DataFrame({'windspeed': windspeed, 'vehicles': vehicles, 'direction': direction, 'pollution': pollution})
df

In [ ]:
def fit_data(df, X, y):
    try:
        lr = LinearRegression()
        lr.fit(X, y)
        rep = f"y = {lr.intercept_:0.2f}"
        for i, coef in enumerate(lr.coef_):
            rep += f" + {coef:0.2f}*{df.columns[i]}"
        return rep
    except Exception as e:
        print(e)
        return None
        

In [ ]:
fit_data(df, df[df.columns[:-1]], df['pollution'])

In [ ]:
# Ordinal encoding
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
enc = OrdinalEncoder()

In [ ]:
df2 = df.copy()
df2['direction'] = enc.fit_transform(df[['direction']]).flatten()
df2

In [ ]:
fit_data(df2, df2[df2.columns[:-1]], df2['pollution'])

In [ ]:
pd.Series({x: i for i, x in enumerate(enc.categories_[0])})

In [ ]:
# One-hot encoding
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False)

In [ ]:
direction_ohe = ohe.fit_transform(df[['direction']])
direction_ohe

In [ ]:
col_names_ohe = [f"Is it {x}?" for x in enc.categories_[0]]

In [ ]:
direction_ohe_df = pd.DataFrame(direction_ohe, columns=col_names_ohe)
direction_ohe_df

In [ ]:
# Confirm that we can write Is it W? as a linear combination of the other columns
1-direction_ohe_df[["Is it N?", "Is it S?", "Is it E?"]].sum(axis=1) - direction_ohe_df["Is it W?"]


In [ ]:
X = np.hstack([df[['windspeed', 'vehicles']].values, direction_ohe])

In [ ]:
X

In [ ]:
X_aug = np.hstack([np.ones((X.shape[0], 1)), X])

In [ ]:
X_aug

In [ ]:
X_aug.shape

In [ ]:
np.linalg.matrix_rank(X_aug), np.linalg.matrix_rank(X_aug.T @ X_aug), (X_aug.T @ X_aug).shape

In [ ]:
pd.DataFrame(X_aug.T @ X_aug)

In [ ]:
ohe = OneHotEncoder(sparse_output=False, drop='first')
ohe.fit_transform(df[['direction']])


In [ ]:
direction_ohe_n_1 = ohe.fit_transform(df[['direction']])
col_names_ohe_n_1 = [f"Is it {x}?" for x in enc.categories_[0][1:]]
df_ohe_n_1 = pd.DataFrame(direction_ohe_n_1, columns=col_names_ohe_n_1)
df_ohe_n_1

In [ ]:
X = np.hstack([df[['windspeed', 'vehicles']].values, df_ohe_n_1.values])
X_aug = np.hstack([np.ones((X.shape[0], 1)), X])

X_aug

In [ ]:
np.linalg.matrix_rank(X_aug), np.linalg.matrix_rank(X_aug.T @ X_aug), (X_aug.T @ X_aug).shape

In [ ]:
# Interepeting dummy variables

## dataset

X = np.array(['F', 'F', 'F', 'M', 'M'])
y = np.array([5, 5.2, 5.4, 5.8, 6])

In [ ]:
from sklearn.preprocessing import LabelBinarizer
l = LabelBinarizer()
l.fit_transform(X)

In [ ]:
X_binary = 1 - l.fit_transform(X)

In [ ]:
X_binary    

In [ ]:
lr = LinearRegression()
lr.fit(X_binary, y)

In [ ]:
lr.coef_, lr.intercept_

In [ ]:
y[(X_binary==0).flatten()].mean()

In [ ]:
y[(X_binary==1).flatten()].mean()